# Lab 4 — Clean the broken sensor feed
### EP0418 · Topic 4 · 2 practical hours

**You need:** nothing to download: the notebook fetches `ep0418.py` and
`smart_building_sensors.csv` itself. Have the printed Lab 4 sheet beside you.

## How the cells work

- **Run every grey code cell when you reach it**, top to bottom: click in it and
  press Shift+Enter. A step tells you if a cell needs something first.
- **Text boxes** are the cells with a bold label ending in a colon, like
  **My answers:**. Double-click one to type in it, then press Shift+Enter. You may
  write on the printed sheet instead.
- **Getting code from Gemini.** Each task gives you a prompt to paste into Gemini.
  Gemini puts its code into the notebook as a **green cell**: click **Accept and
  run** (bottom of the chat panel). Then run the check cell under the task. Its
  first line, starting `# TODO(you):`, says what must run first. Lines like
  `s = keep("s")` pick up what Gemini's cell made: do not edit them. If the check
  cell says *Not there yet* (or *No ... yet*), Gemini used a different name: type in
  the chat *call the result* followed by the name the task asks for, accept the new
  cell, and run the check cell again.
- **Fill-in cells** have lines like `TEAM = ""` or `H = None` for you to type into:
  replace the `""` or the `None`, keeping the rest of the line. Type numbers
  without commas (`128000`, not `128,000`). An example is always shown.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

Every prompt in these notebooks opens with the rules that matter for that task,
such as do not drop rows, show the row count before and after, or explain the code
in plain words. Keep those lines when you adapt a prompt, and start your own
prompts the same way.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.


> ### This dataset is deliberately broken
> **Six** separate faults are hidden in this file: −999 values that are not
> readings, duplicate timestamps, a five-hour outage, an impossible energy spike, a
> sensor that gets stuck, and a sensor that slowly drifts. The table and the checks
> show you the first four. **The last two you will only find by plotting**, in
> Task 5.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

> **Then run the grey code cell just below** (click in it and press Shift+Enter, or click the play button at its left). Colab may warn that the notebook was not authored by Google: click **Run anyway**. You should see nothing in red; lines like *fetched ...*, a row count or a short table are normal. That means the course files and tools are loaded. Every lab starts this way. From here on, **run every grey code cell when you reach it**, top to bottom. **To write in a text box** (a cell with a bold label ending in a colon), double-click it, type, then press Shift+Enter.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def keep(name):
    """The value Gemini's cell already made under this name, or ... if there is none yet."""
    return globals().get(name, ...)


def filled(*values):
    """True once every TODO above has a value. Otherwise says what to do."""
    if any(v is ... or v is None for v in values):
        print("Not there yet. Run Gemini's code first (click Accept and run on its green cell, "
              "or paste its code into a new cell and run it), then run this cell again.")
        return False
    return True


def submit_prediction(team, lab, prediction, method):
    """Send one team's number to the Prediction League board."""
    import urllib.parse
    form = "https://docs.google.com/forms/d/e/1FAIpQLSdwpzHafC_rmPDdbK_cm6J8CihD-NSVS7ZBUyYCWIrS_MMJ0g/formResponse"
    fields = {'team': 'entry.911171418', 'lab': 'entry.1890998159', 'prediction': 'entry.959891957', 'method': 'entry.1997288724'}
    if not team or not method or prediction is None:
        print("Fill in TEAM, MY_PREDICTION and METHOD above, then run this cell again."); return
    entry = {"team": team, "lab": str(lab), "prediction": f"{float(prediction):.1f}", "method": method}
    if not form:
        print("The Prediction League is not switched on yet; tell your tutor your number."); return
    data = urllib.parse.urlencode({fields[k]: v for k, v in entry.items()}).encode()
    try:
        urllib.request.urlopen(urllib.request.Request(form, data=data, headers={"User-Agent": "EP0418"}), timeout=60)
        print(f"Entered: {team} predicts {float(prediction):.1f}. Check the scoreboard in a minute.")
    except Exception as e:
        print("The league did not answer:", e, "- try again in a minute.")

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows, {raw.timestamp.min()} -> {raw.timestamp.max()}")

In [ ]:
print("blank values:", raw.isna().sum().sum())
print("duplicated timestamps:", raw.timestamp.duplicated().sum())
print()
print(raw.describe().T[["min", "mean", "max"]].round(1).to_string())

### Read that table before going further

**What you should see:** 0 blank values, 140 duplicated timestamps, and a table of
minimum, mean and maximum per column. `isna()` reports zero blanks. But look at the
minimum for `humidity_pct`.

**−999 is not a reading.** It is the logger saying "no answer". pandas counts it
as a real number, so it is dragging your mean down right now and `isna()` will
never find it.

1. **Compare** each maximum in the table with its mean.
2. **Find** any other column with a minimum or maximum that is physically
   impossible. Hint: a classroom does not use eight times its usual energy in five
   minutes.
3. **Write it** in the box below. You do not fix it today: Lab 5 is about spikes.

**Other impossible values (column, min or max, why impossible):**

## Task 1 — Replace the sentinels

You want `clean`: a copy of `raw` where each −999 in `humidity_pct` is blank (NaN),
with every row kept.

**Step 1. Ask Gemini.** Open Gemini (the **spark icon**, bottom right of Colab), click
**+** in the chat box for a new chat, paste this and press Enter:

> Do not remove any rows. Show the row count before and after. Explain in plain
> words what the code does. I have a DataFrame `raw` where -999 in `humidity_pct`
> means "no reading". Make a copy called `clean` with those -999 values replaced by
> NaN.

**Step 2. Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it.

**Step 3. Run the check cell below.** You should see two `[PASS]` lines, and 17,354 rows (the expected output is under the cell).

**Step 4. Only if a check fails or a number looks wrong:** in the same chat, type one
sentence saying what you saw and what you expected, for example *clean has 17,259 rows but must keep all 17,354; replace -999 with NaN instead of removing the rows.* Press Enter,
click **Accept and run** on the new green cell, and run the check cell below again. If everything
passed, there is nothing to fix: move on.

A common wrong answer filters the rows out instead of marking them blank, which
changes the row count; `rows_at_least` catches it.

In [ ]:
# TODO(you): Accept and run Gemini's green cell above first (it makes `clean`), then run this cell.
clean = keep("clean")

# ---- verification contract — do not edit ----
check.no_sentinels(clean, cols=["humidity_pct"], sentinel=-999)
check.rows_at_least(clean, 17000)
check.report()

**Expected output** *(computed from the data file when this notebook was built)*: still 17,354 rows; 95 values of −999 became NaN. If the row count fell, rows were dropped rather than values replaced.

**Run the next cell.** **Look for** what changed between the two panels, and what
did not.

In [ ]:
if filled(clean):
    show.before_after(raw, clean, "humidity_pct", title="Replacing -999 with NaN")

**What you should see:** the top panel has the vertical spikes to −999; the bottom
panel has small gaps in the line where they were. The row count is the same in
both titles. Nothing was removed: the failures are now shown as *missing* instead
of as a number, which is what they always were.

## Task 2 — Duplicates and time order

Some timestamps appear twice: the logger retried. On a logger that retries, the
last copy is the corrected one, so you keep the last copy and drop the others.

**Step 1. Ask Gemini.** Open Gemini (the **spark icon**, bottom right of Colab), click
**+** in the chat box for a new chat, paste this and press Enter:

> Do not drop any rows except the duplicate copies. Show the row count before and
> after. Explain in plain words what the code does. `clean` is a DataFrame with a
> `timestamp` column, and some timestamps appear more than once. Remove the
> duplicates, keeping the last copy of each timestamp, sort by timestamp, and keep
> the result in `clean`.

**Step 2. Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it.

**Step 3. Run the check cell below.** You should see two `[PASS]` lines, and 17,214 rows (140 fewer).

**Step 4. Only if a check fails or a number looks wrong:** in the same chat, type one
sentence saying what you saw and what you expected, for example *sorted_in_time failed; sort clean by timestamp after removing the duplicates.* Press Enter,
click **Accept and run** on the new green cell, and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell above first (it updates `clean`), then run this cell.
clean = keep("clean")

# ---- verification contract — do not edit ----
check.no_duplicate_timestamps(clean, col="timestamp")
check.sorted_in_time(clean, col="timestamp")
check.report()
if filled(clean):
    print(f"rows in clean: {len(clean):,}")

**Expected output** *(computed from the data file when this notebook was built)*: 17,354 rows become 17,214: 140 duplicate timestamps removed. In the Decision box at the end, write: kept the last copy · the logger retries, so the last copy is the corrected one · rows 17,354 → 17,214.

## Task 3 — Put it on a regular grid

**Resample** means: put the readings onto a fixed timetable, one row per 5-minute
slot, even for slots where the sensor sent nothing (those stay blank). You want
`grid`.

**Step 1. Ask Gemini.** Open Gemini (the **spark icon**, bottom right of Colab), click
**+** in the chat box for a new chat, paste this and press Enter:

> Do not drop empty slots: leave them as NaN. Show the row count before and after.
> Explain in plain words what the code does. `clean` is a DataFrame with a
> `timestamp` column and the columns temp_c, humidity_pct, co2_ppm, energy_kwh and
> occupancy. Set timestamp as the index and resample onto a regular 5-minute grid,
> taking the mean in each slot. Call the result `grid`.

**Step 2. Accept its code.** Gemini explains its plan in the chat and puts its code
into the notebook as a **green cell**. Click **Accept and run**: the button sits at
the bottom of the chat panel, just above the box where you type. It does not matter
where in the notebook the green cell lands. Ignore the blue suggestion chips under
its answer. If the code appears only in the chat, click the copy icon on the code,
click once on this text, click **+ Code** at the top of the page, paste into the new
cell, and run it.

**Step 3. Run the check cell below.** You should see two `[PASS]` lines, and 17,280 rows: 66 *more* than `clean`.

**Step 4. Only if a check fails or a number looks wrong:** in the same chat, type one
sentence saying what you saw and what you expected, for example *regular_cadence failed: grid is not every 5 minutes; resample with '5min' on the timestamp index.* Press Enter,
click **Accept and run** on the new green cell, and run the check cell below again. If everything
passed, there is nothing to fix: move on.

In [ ]:
# TODO(you): Accept and run Gemini's green cell above first (it makes `grid`), then run this cell.
grid = keep("grid")

# ---- verification contract — do not edit ----
check.regular_cadence(grid, "5min")
check.values_in_range(grid.reset_index(), "temp_c", 15, 40)
check.report()
if filled(grid):
    print(f"rows in grid: {len(grid):,}")

**Expected output** *(computed from the data file when this notebook was built)*: 17,214 rows become 17,280 on the grid: the count goes *up* by 66, the empty slots of the 11 June outage, which now exist as rows of NaN.

**Run the next cell.** **Look for** the shaded hours on 11 June.

In [ ]:
if filled(grid):
    show.series(grid, col="co2_ppm", title="CO2 on the regular 5-minute grid; shaded: the 11 June outage",
                span=("2026-06-11 02:00", "2026-06-11 07:30"))

**What you should see:** a break in the line inside the shaded period. Before
resampling, those hours simply had no rows and nothing showed. Now they are empty
slots on the grid, and the chart is honest about them. Count them in Task 4: the
row count goes *up* from `clean` to `grid`, and you have to say why.

## Task 4 — Account for every change in the row count

Task 1 kept every row. The count then changed twice: in Task 2 (`raw` 17,354 →
`clean` 17,214) and in Task 3 (`clean` 17,214 → `grid` 17,280).

1. **In the cell below**, replace the words between the quotes after `EXPLANATION =`
   with your own explanation of both changes: **which** rows and **why**. For example
   (made-up numbers): *"12 rows removed in Task 2 because ...; 30 rows
   added in Task 3 because ..."*. *"Cleaning removed some rows"* will be rejected.
2. **Run the cell.** You should see `[PASS] rows_accounted_for`. If it says your
   explanation is missing or too short, add the reasons and run it again.

In [ ]:
# TODO(you): replace the words between the quotes with your explanation (see the shape above)
EXPLANATION = "TODO(you): replace with a real explanation"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw, clean, grid,
    names=["raw", "clean", "grid"],
    explain=EXPLANATION,
)
check.report()

## Task 5 — Find the two faults no check can catch

1. **Run the cell below.** It draws all four sensor channels over the sixty days.
2. **Look for** the two faults described under it. Sixty days on one axis can hide
   a fourteen-hour flat line, so if you cannot see it, ask Gemini in a new chat:
   *For each column of `grid`, find the longest run of identical consecutive
   values and the date it starts.* Accept and run its cell.
3. **Write** what you found in the box at the end of this task.

One fault is a sensor that **stops changing** — it produces no outliers at all,
just a flat line, which looks perfectly well-behaved to any alarm.

The other is a **slow drift** that makes a fixed threshold fire more and more
often even though the room never changed.

In [ ]:
# Draw all four channels. The finding is yours; the drawing is not the exercise.
if filled(grid):
    for col in ["temp_c", "humidity_pct", "co2_ppm", "energy_kwh"]:
        show.series(grid, col=col, title=col)

**Look for:** in one channel, a stretch that is perfectly flat; in another, a slow
climb over the last three weeks that never comes back down. Neither is out of
range, neither is a duplicate, neither is missing. Every check in the library
passes on both.

**Stuck sensor (channel, dates, what it was doing):**

**Drifting sensor (channel, dates, what it was doing):**

---
## Predict: How many bone-dry days next week? (10 minutes)

*The Prediction League: Lab 4's question.*

**The question:** On how many of the seven days of the target week will the island-average rain be zero?
One number per team. **Teams:** two or three people, the same team and name every
week; if you have no team yet, join one or two neighbours now and agree a name. **The scoreboard** is at
https://stanleysp-eng.github.io/EP0418-labs/ (open it on a phone): it shows the
target week's dates, every team's number the moment it is sent, and the real
number when the week is over. Nearest team wins the week.

**Step 1. Run the cell below.** It prints two candidate numbers, A and B, and the
exact question to ask Gemini.

In [ ]:
fetch("sg_rainfall_daily.csv")
rain = pd.read_csv("sg_rainfall_daily.csv", parse_dates=["date"])
dry = rain.rain_mm_mean == 0
A = round(dry.mean() * 7)                      # the file's share of dry days, over a week
B = round(dry.tail(14).sum() / 2)              # the last fortnight's rate, over a week
print(f"A. the sixty-day share of dry days, as a week : {A} days   <- the baseline on the board")
print(f"B. the last fortnight's rate, as a week       : {B} days")
print(f"   ({int(dry.sum())} dry days in {len(rain)}; {int(dry.tail(14).sum())} in the last fortnight)")

print()
print("Step 2. Open Gemini (spark icon, bottom right), press + for a new chat, and paste this:")
print()
print(f"I am new to this. Answer in three plain sentences, no headings, no code. I must predict how many of the seven days of the target week will be bone dry (zero island-average rain). "
      f"One estimate says {A:.0f}, based on the share of dry days over the whole sixty days; another says {B:.0f}, based on the last fortnight. "
      "Which should I use, or a number in between, and why might it still be wrong?")

**Step 2. Ask Gemini.** Copy the printed question: drag across it in the output,
from *I am new* to *wrong?*, and press Ctrl+C (Cmd+C on a Mac). Open Gemini, click
**+** for a new chat, paste it and press Enter.
Read the three sentences. It will pick A, B or something between and give a reason.
Ignore the blue suggestion chips under its answer. If it writes an essay, type
*simpler, three sentences*.

**Step 3. Type your number and send it.** Talk it over with your team: Gemini's
pick, or your own. **One person per team** puts it in the cell below, with the team
name and one line on why, and runs the cell; the others skip that cell. Filled in, it looks like this:

```
TEAM = "Team Rocket"
MY_PREDICTION = 1
METHOD = "Gemini said B; we nudged it because ..."
```

The message *Entered: Team Rocket predicts ...* confirms it. Open the scoreboard
(https://stanleysp-eng.github.io/EP0418-labs/) and look for your team under
"Prediction League" within a minute. Nobody is marked on the method; only reality
marks the number.

In [ ]:
TEAM = ""                 # TODO(you): your team name
MY_PREDICTION = None      # TODO(you): your number, no commas, e.g. 1
METHOD = ""               # TODO(you): one line on how you chose it

submit_prediction(TEAM, 4, MY_PREDICTION, METHOD)

**Write it down** (on the sheet, or double-click here): your number, the baseline,
and the one reason you moved away from the baseline, if you did.

**Our number / the baseline / why:**

---
## Read before you run

An assistant was asked to put the cleaned sensor data onto a 5-minute grid with no blank values left. It wrote the code below.

The code is in the cell below, commented out. It came from an AI assistant in
someone else's chat; you are not asking Gemini for it.

1. **Read it. Do not run it.** Answer questions 1 and 2 (just below) in the text
   box under the code cell (double-click it to type), or on the sheet.
2. **Ask Gemini to explain it.** Click once in the code cell. At its top right,
   click the icon that looks like a pencil with a small star (Colab calls it the
   Gemini icon) and choose **Explain code**. Cannot find it? Open the chat instead
   (spark icon, **+** for a new chat), paste the code, and ask *Explain this code in
   plain words.* Read the explanation. Did it mention the fault? It often does not.
3. **Only then run it.** Delete the `# ` at the start of every line *below the
   first line* (the first line is the instruction and stays), then run the cell
   and see whether you were right.

**Question 1.** What does the `interpolate()` line do to the five-hour outage on 11 June? Would the Task 3 checks (`regular_cadence`, `values_in_range`) fail?

**Question 2.** Predict the two printed blank counts, and whether the Task 3 checks pass. Then run it: it prints both counts and runs those two checks on its result. (It uses its own name, `their_grid`, so your `grid` is untouched.)

<details><summary>Hint 1: where to look</summary>

What does `interpolate()` do to a run of empty slots between two real readings?

</details>
<details><summary>Hint 2: closer to the answer</summary>

It draws a straight line through them. The 66 empty slots of the 11 June outage get invented values, and so does every blank humidity value. The row count does not change, so `rows_accounted_for` cannot see it, and the Task 3 checks still pass.

</details>

An explanation that reads well is not evidence the code is right; that is the
point of step 2. Running this code may change variables from your own tasks; if a
later cell misbehaves, run your own Gemini cell from the task again.

**Optional, to fix it:** click the same Gemini icon on the cell, choose **Transform
code**, and say in plain words what to change (your answer to question 1). Colab
shows the change as a diff before it is applied. Read the diff before you accept it.

In [ ]:
# ---- Every line below this one is the code. Do steps 1 and 2 first. For step 3, delete the "# " at the start of each line below, then run the cell. ----
# their_grid = clean.set_index("timestamp").resample("5min").mean()
# print("blank values before interpolate:", int(their_grid.isna().sum().sum()))
# their_grid = their_grid.interpolate()
# print("blank values after interpolate :", int(their_grid.isna().sum().sum()))
# check.reset(); check.regular_cadence(their_grid, "5min"); check.values_in_range(their_grid.reset_index(), "temp_c", 15, 40); check.report()

**Answer to question 1:**

**Answer to question 2:**

**Did Gemini's explanation mention the fault? (yes / no, and the sentence):**

**Was I right, once I ran it?**

---
## Quiz yourself before you leave (5 minutes)

The checks tell you whether the code is right. They do not tell you whether
**you** know what it did. Open Gemini, click **+** for a new chat, paste this, and
press Enter:

> Quiz me on this lab. Ask five short questions, one at a time, in plain words, on these ideas: the four disguises of a missing value; why duplicate timestamps appear and which copy to keep; why every row you remove must be accounted for; what rows_accounted_for checks and what it cannot see; what a stuck sensor looks like and why no check catches it. Take the examples from the code and outputs in this notebook. Wait for my answer. Then say in one sentence whether I am right, and why. Do not give the next question until I answer. After the fifth, give my score out of 5 and the one idea I should revise, in one line.

Gemini asks in the chat panel; you answer in the chat box; it marks you and asks
the next. Ignore the suggestion chips under its messages. Answer in your own
words, without looking back until it has marked you.

Then write your score and the idea to revise below. Your tutor asks for them in
the plenary.

**Quiz score:** ___ / 5

**The idea I should revise, in my own words:**

---
## Before you leave

Double-click the box cell below and type there (or write on the last page of the Lab 4
sheet): one **mistake caught** line per AI mistake you caught today, and one
**decision** line for each fault you handled (the −999s, the duplicates, the outage
gap), saying what you did and why. The Task 2 expected output gives one example.

**Mistake caught:** *(what I expected · what was there · how I found it)*

**Decision:** *(what I decided · why · the evidence)*